# 04D — GraphRAG and Advanced Architectures

Building on the retrieval trilogy. Where prior notebooks fixed *what* you retrieve and *how* you evaluate it, this one asks a different question: what if the answer isn't *in* a document — it's *between* documents?

**Sections:**
1. Why vector search fails on relationship queries
2. Entity and relationship extraction
3. Build the knowledge graph
4. Graph traversal vs vector retrieval
5. Entity resolution demonstration
6. Corrective RAG simulation
7. Agentic RAG with tool use
8. When to use each architecture
9. Key observations

In [ ]:
%pip install anthropic networkx matplotlib pyvis

In [ ]:
import anthropic
import json
import os
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic()

print(f"Model: {MODEL}")
print(f"NetworkX version: {nx.__version__}")

---
## Section 1 — Why Vector Search Fails on Relationship Queries

Vector search ranks documents by similarity to the query. Relationship queries ask something different: *how are these two things connected?* These are different questions, and the same retrieval mechanism answers one well and the other poorly.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Mini FinMentor corpus — 6 chunks
corpus = [
    {
        "id": "aapl_pos",
        "text": "AAPL position: 50 shares at avg cost $182.40. Current price $211.75. "
                "AAPL belongs to the Technology sector. P&L +16.1%."
    },
    {
        "id": "msft_pos",
        "text": "MSFT position: 30 shares at avg cost $378.20. Current price $425.30. "
                "MSFT belongs to the Technology sector. P&L +12.4%."
    },
    {
        "id": "fed_rate",
        "text": "Federal Reserve announces 25bps rate hike. Higher rates increase discount rates "
                "applied to future cash flows, compressing valuations of growth stocks."
    },
    {
        "id": "tech_sector",
        "text": "Technology sector valuation report: sector P/E ratio 28x. Highly sensitive to "
                "interest rate changes due to growth-stock duration. Rising rates compress multiples."
    },
    {
        "id": "gs_nvda",
        "text": "Goldman Sachs initiates NVDA with Buy rating and $1,050 price target. "
                "Blackwell demand ahead of estimates. NVDA is a Technology sector company."
    },
    {
        "id": "pnl_summary",
        "text": "Portfolio P&L summary: total unrealized gain $38,215. Realized P&L $4,820. "
                "Technology positions account for 82% of unrealized gains."
    }
]

corpus_texts = [c["text"] for c in corpus]
corpus_ids   = [c["id"]   for c in corpus]

# TF-IDF embeddings — no API needed
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_texts)

def vector_search(query: str, top_k: int = 3):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix)[0]
    ranked = sorted(zip(corpus_ids, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

rel_queries = [
    "How are AAPL and MSFT related in terms of sector risk?",
    "How does Fed policy affect my tech positions?"
]

print("=" * 70)
print("VECTOR SEARCH — relationship queries")
print("=" * 70)
for q in rel_queries:
    results = vector_search(q, top_k=3)
    print(f"\nQ: {q}")
    for doc_id, score in results:
        text = next(c["text"] for c in corpus if c["id"] == doc_id)
        print(f"  [{doc_id:<12}] score={score:.3f}  {text[:70]}...")
    print(f"  Missing: no explicit AAPL↔MSFT link or Fed→Tech→positions chain assembled")

Vector search finds documents *similar* to the query. It cannot assemble a *chain of relationships* — AAPL → Tech Sector ← MSFT, or Fed Rate → Tech Sector Valuation → Portfolio Positions. The relationship lives between documents, not inside any one of them. That's what a knowledge graph stores.

---
## Section 2 — Entity and Relationship Extraction

Building the graph starts with reading the corpus and extracting what's in it: the entities (things) and the relationships between them.

In [ ]:
EXTRACTION_SYSTEM = """\
Extract entities and relationships from the financial text.
Return JSON only — no markdown.

Schema:
{
  "entities": [{"name": str, "type": str, "description": str}],
  "relationships": [{"source": str, "target": str, "type": str, "description": str}]
}

Entity types: COMPANY, SECTOR, METRIC, PERSON, EVENT, CONCEPT
Relationship types: belongs_to, correlated_with, affects, held_by,
  reported_by, rates_buy, rates_sell, sensitive_to, part_of

Use canonical names: "Goldman Sachs" not "GS", "AAPL" not "Apple Inc".
Be specific: every relationship needs both a source and a target that
appear in the entities list."""


def extract_graph_elements(document: str) -> dict:
    raw = client.messages.create(
        model=MODEL,
        max_tokens=600,
        system=EXTRACTION_SYSTEM,
        messages=[{"role": "user", "content": document}]
    ).content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)


print("Extracting entities and relationships from 6 corpus chunks...\n")
all_extractions = []
total_entities, total_rels = 0, 0

for chunk in corpus:
    extraction = extract_graph_elements(chunk["text"])
    extraction["source_id"] = chunk["id"]
    all_extractions.append(extraction)
    n_e = len(extraction["entities"])
    n_r = len(extraction["relationships"])
    total_entities += n_e
    total_rels += n_r
    print(f"[{chunk['id']:<12}] {n_e} entities, {n_r} relationships")
    for e in extraction["entities"]:
        print(f"   entity : {e['name']} ({e['type']})")
    for r in extraction["relationships"]:
        print(f"   rel    : {r['source']} --{r['type']}--> {r['target']}")
    print()

print(f"Total extracted: {total_entities} entities, {total_rels} relationships")

---
## Section 3 — Build the Knowledge Graph

Merge all extractions into a single directed graph. Duplicate entity references get merged into one node — that's where the cross-document connection is made.

In [ ]:
TYPE_COLORS = {
    "COMPANY":  "#2a9d8f",   # teal
    "SECTOR":   "#457b9d",   # blue
    "CONCEPT":  "#adb5bd",   # gray
    "EVENT":    "#e76f51",   # coral
    "METRIC":   "#f4a261",   # orange
    "PERSON":   "#a8dadc",   # light teal
}
DEFAULT_COLOR = "#ced4da"


def build_knowledge_graph(extractions: list[dict]) -> nx.DiGraph:
    G = nx.DiGraph()
    # Collect all entity metadata first
    entity_meta: dict[str, dict] = {}
    for extraction in extractions:
        for e in extraction["entities"]:
            name = e["name"]
            if name not in entity_meta:
                entity_meta[name] = {"type": e["type"], "description": e["description"],
                                      "sources": []}
            entity_meta[name]["sources"].append(extraction["source_id"])
    # Add nodes
    for name, meta in entity_meta.items():
        G.add_node(name, node_type=meta["type"], description=meta["description"],
                   sources=meta["sources"],
                   color=TYPE_COLORS.get(meta["type"], DEFAULT_COLOR))
    # Add edges
    for extraction in extractions:
        for r in extraction["relationships"]:
            src, tgt = r["source"], r["target"]
            if src in G and tgt in G:
                G.add_edge(src, tgt, relationship=r["type"],
                           description=r.get("description", ""),
                           source_doc=extraction["source_id"])
    return G


G = build_knowledge_graph(all_extractions)

# Graph stats
from collections import Counter
type_counts = Counter(nx.get_node_attributes(G, "node_type").values())
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)

print(f"Nodes : {G.number_of_nodes()}")
print(f"Edges : {G.number_of_edges()}")
print(f"Node types: {dict(type_counts)}")
print(f"Most connected nodes (top 3):")
for node, degree in degree_sorted[:3]:
    ntype = G.nodes[node].get("node_type", "?")
    print(f"  {node} ({ntype}) — degree {degree}")

# Matplotlib visualization
fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=2.0)
node_colors = [G.nodes[n].get("color", DEFAULT_COLOR) for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1200, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold", ax=ax)
nx.draw_networkx_edges(G, pos, edge_color="#6c757d", arrows=True,
                       arrowsize=15, connectionstyle="arc3,rad=0.1", ax=ax)
edge_labels = {(u, v): d["relationship"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6, ax=ax)

# Legend
legend_patches = [plt.matplotlib.patches.Patch(color=c, label=t)
                  for t, c in TYPE_COLORS.items() if t in type_counts]
ax.legend(handles=legend_patches, loc="upper left", fontsize=8)
ax.set_title("FinMentor Knowledge Graph", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.savefig("finmentor_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph saved to finmentor_graph.png")

---
## Section 4 — Graph Traversal vs Vector Retrieval

The same two queries that failed in Section 1 now get answered by walking the graph instead of scoring documents.

In [ ]:
def graph_traverse(query_entity: str, G: nx.DiGraph, max_hops: int = 2) -> list[dict]:
    if query_entity not in G:
        # Fuzzy match: find a node whose name contains the query entity
        matches = [n for n in G.nodes() if query_entity.lower() in n.lower()]
        if not matches:
            return []
        query_entity = matches[0]

    context = []
    visited = {query_entity}
    queue   = [(query_entity, 0)]

    while queue:
        node, depth = queue.pop(0)
        if depth >= max_hops:
            continue
        for neighbor in list(G.successors(node)) + list(G.predecessors(node)):
            if G.has_edge(node, neighbor):
                edge = G[node][neighbor]
                direction = "→"
            else:
                edge = G[neighbor][node]
                direction = "←"
            context.append({
                "from":         node,
                "direction":    direction,
                "relationship": edge["relationship"],
                "to":           neighbor,
                "description":  edge.get("description", ""),
                "depth":        depth + 1
            })
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, depth + 1))
    return context


traversal_queries = [
    ("AAPL",         "How are AAPL and MSFT related in terms of sector risk?"),
    ("Federal Reserve", "How does Fed policy affect my tech positions?"),
]

print("=" * 70)
print("GRAPH TRAVERSAL — same relationship queries")
print("=" * 70)
for start_entity, query in traversal_queries:
    path = graph_traverse(start_entity, G, max_hops=2)
    print(f"\nQ: {query}")
    print(f"Starting from: '{start_entity}'")
    if path:
        for step in path:
            indent = "  " * step["depth"]
            print(f"{indent}[hop {step['depth']}] {step['from']} {step['direction']} "
                  f"{step['relationship']} {step['direction']} {step['to']}")
            if step["description"]:
                print(f"{indent}         {step['description'][:80]}")
    else:
        print("  No paths found — entity not in graph")

# Side-by-side comparison
print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"{'Query':<45} {'Vector result':<20} {'Graph result'}")
print("-" * 90)
comparisons = [
    ("AAPL↔MSFT sector risk",         "aapl_pos + msft_pos",  "AAPL→Tech Sector←MSFT (shared node)"),
    ("Fed policy → tech positions",   "fed_rate chunk only",  "Fed Rate→Tech Sector→Portfolio Positions"),
]
for q, vec, gph in comparisons:
    print(f"{q:<45} {vec:<20} {gph}")

Graph traversal assembles the connection automatically from structure, not from text similarity. The AAPL and MSFT documents never mention each other — but they both point to the Technology Sector node. One hop finds the shared risk. Vector search would need both documents to appear similar to the query *and* would still return two separate chunks with no explicit connection stated.

---
## Section 5 — Entity Resolution Demonstration

The same firm referred to in three different ways across three documents. Without resolution, you get three nodes. With resolution, you get one node with three source references — and every query that finds one of them finds all of them.

In [ ]:
# Three chunks each using a different name for the same firm
noisy_chunks = [
    {"id": "gs_a", "text": "GS Equity Research recommends NVDA. Price target raised to $1,100."},
    {"id": "gs_b", "text": "Goldman analysts raise semiconductor sector outlook to Overweight."},
    {"id": "gs_c", "text": "GS research team sees significant upside in AI infrastructure spending."},
]

print("Extracting entities WITHOUT resolution:")
noisy_extractions = []
raw_entity_names = []
for chunk in noisy_chunks:
    ext = extract_graph_elements(chunk["text"])
    ext["source_id"] = chunk["id"]
    noisy_extractions.append(ext)
    for e in ext["entities"]:
        if any(kw in e["name"].lower() for kw in ["gs", "goldman", "sachs"]):
            raw_entity_names.append(e["name"])
            print(f"  [{chunk['id']}] extracted entity: '{e['name']}' ({e['type']})")

print(f"\nWithout resolution: {len(set(raw_entity_names))} distinct entity name(s) for the same firm: {set(raw_entity_names)}")


def resolve_entities(entity_names: list[str]) -> dict[str, str]:
    """Ask Claude which entity names refer to the same thing and return a canonical mapping."""
    prompt = (
        "The following entity names were extracted from different documents. "
        "Identify which ones refer to the same real-world entity and provide a canonical name. "
        "Return JSON only: {\"mappings\": [{\"aliases\": [list], \"canonical\": str}]}"
        f"\n\nEntities: {json.dumps(entity_names)}"
    )
    raw = client.messages.create(
        model=MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    mappings = json.loads(raw)["mappings"]
    alias_to_canonical = {}
    for group in mappings:
        for alias in group["aliases"]:
            alias_to_canonical[alias] = group["canonical"]
    return alias_to_canonical


# Collect all entity names across all noisy chunks for resolution
all_noisy_entity_names = list({e["name"] for ext in noisy_extractions for e in ext["entities"]})
resolution_map = resolve_entities(all_noisy_entity_names)

print("\nEntity resolution map:")
for alias, canonical in resolution_map.items():
    if alias != canonical:
        print(f"  '{alias}' → '{canonical}'")

# Apply resolution and rebuild mini graph
def apply_resolution(extractions: list[dict], resolution_map: dict) -> list[dict]:
    resolved = []
    for ext in extractions:
        new_entities = [
            {**e, "name": resolution_map.get(e["name"], e["name"])}
            for e in ext["entities"]
        ]
        new_rels = [
            {**r,
             "source": resolution_map.get(r["source"], r["source"]),
             "target": resolution_map.get(r["target"], r["target"])}
            for r in ext["relationships"]
        ]
        resolved.append({**ext, "entities": new_entities, "relationships": new_rels})
    return resolved

resolved_extractions = apply_resolution(noisy_extractions, resolution_map)
G_resolved = build_knowledge_graph(resolved_extractions)

# Check Goldman Sachs node
gs_canonical = resolution_map.get("Goldman Sachs", "Goldman Sachs")
# Find whichever node has 'goldman' or 'gs' in it
gs_nodes = [n for n in G_resolved.nodes() if "goldman" in n.lower() or (n.upper() == "GS")]

print("\nAfter resolution:")
for gs_node in gs_nodes:
    sources = G_resolved.nodes[gs_node].get("sources", [])
    print(f"  Node: '{gs_node}'")
    print(f"  Sources: {sources}  (all 3 chunks → 1 node)")
    print(f"  Edges: {list(G_resolved.edges(gs_node))}")

Entity resolution is a one-time cost at index time. At query time, every reference to Goldman — "GS," "Goldman analysts," "GS research team" — hits the same node and surfaces all three source documents automatically. Vector search never solves this cleanly: the three strings have different embeddings, and under competition with other documents the right one may not surface.

---
## Section 6 — Corrective RAG Simulation

Standard RAG generates from whatever it retrieved, regardless of retrieval quality. CRAG assesses retrieval quality first and decides whether to generate, filter, or refuse.

In [ ]:
def assess_relevance(query: str, chunks: list[str]) -> list[float]:
    """Score each chunk 0.0–1.0 for relevance to query."""
    prompt = (
        f"Query: {query}\n\n"
        "Rate each chunk's relevance to the query on a scale of 0.0 to 1.0.\n"
        "1.0 = directly answers the query. 0.0 = completely irrelevant.\n"
        f"Return JSON only: {{\"scores\": [list of {len(chunks)} floats]}}"
        "\n\nChunks:\n" + "\n".join(f"{i+1}. {c}" for i, c in enumerate(chunks))
    )
    raw = client.messages.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)["scores"]


def generate_from_chunks(query: str, chunks: list[str]) -> str:
    context = "\n\n".join(chunks)
    return client.messages.create(
        model=MODEL, max_tokens=300,
        system="You are a financial assistant. Answer using ONLY the provided context. Be concise.",
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}]
    ).content[0].text.strip()


def corrective_rag(query: str, chunks: list[str]) -> tuple[str, str]:
    scores = assess_relevance(query, chunks)
    max_score = max(scores)
    scored_chunks = list(zip(chunks, scores))

    if max_score > 0.8:
        answer = generate_from_chunks(query, chunks)
        return "GOOD", answer, scores
    elif max_score > 0.4:
        good_chunks = [c for c, s in scored_chunks if s > 0.4]
        answer = generate_from_chunks(query, good_chunks)
        return "PARTIAL", answer, scores
    else:
        return "POOR", "I don't have reliable information about this in my knowledge base. Please consult a qualified financial advisor.", scores


# Three test scenarios
crag_tests = [
    {
        "scenario": "GOOD: query about a held position",
        "query": "What is my AAPL position and its P&L?",
        "chunks": [
            "AAPL position: 50 shares at avg cost $182.40. Current price $211.75. P&L +16.1%.",
            "Portfolio P&L summary: total unrealized gain $38,215.",
            "Federal Reserve announces 25bps rate hike."
        ]
    },
    {
        "scenario": "PARTIAL: query partially covered",
        "query": "How does my tech sector exposure compare to the S&P 500 benchmark?",
        "chunks": [
            "Technology sector valuation report: sector P/E ratio 28x. Highly sensitive to rate changes.",
            "Portfolio P&L: Technology positions account for 82% of unrealized gains.",
            "Federal Reserve announces 25bps rate hike."
        ]
    },
    {
        "scenario": "POOR: query about stock not in knowledge base",
        "query": "What is the current outlook for Tesla stock?",
        "chunks": [
            "AAPL position: 50 shares at avg cost $182.40.",
            "Goldman Sachs initiates NVDA with Buy rating.",
            "Federal Reserve announces 25bps rate hike."
        ]
    }
]

print("=" * 70)
print("CORRECTIVE RAG — three retrieval quality scenarios")
print("=" * 70)
for test in crag_tests:
    assessment, answer, scores = corrective_rag(test["query"], test["chunks"])
    print(f"\nScenario : {test['scenario']}")
    print(f"Query    : {test['query']}")
    print(f"Scores   : {[f'{s:.2f}' for s in scores]}  max={max(scores):.2f}")
    print(f"Assessment: {assessment}")
    print(f"Answer   : {answer[:200]}")

Standard RAG generates confidently from whatever it retrieved. CRAG knows when it's retrieving poorly and acts accordingly: generate from good context, filter to partial context, or refuse entirely. For a financial AI this is the difference between a useful tool and a liability — the POOR case above produces a clear refusal instead of a plausible-sounding wrong answer about Tesla.

---
## Section 7 — Agentic RAG with Tool Use

In agentic RAG, retrieval is not automatic — it's a decision. The agent chooses when to retrieve, what to retrieve, and from where, based on the question.

In [ ]:
# Tool definitions for Claude's tool-use API
TOOLS = [
    {
        "name": "search_portfolio",
        "description": "Search the user's IBKR portfolio data for positions, P&L, and account information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query":      {"type": "string", "description": "What to search for"},
                "date_range": {"type": "string", "description": "Optional: YYYY-MM to YYYY-MM"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "search_market_news",
        "description": "Search analyst reports and market news. Use for analyst ratings, price targets, sector outlooks.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query":   {"type": "string", "description": "What to search for"},
                "tickers": {"type": "array",  "items": {"type": "string"}, "description": "Optional: filter by tickers"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "traverse_knowledge_graph",
        "description": "Traverse the knowledge graph starting from an entity. Use to find relationships between entities — e.g., how Fed policy connects to specific holdings.",
        "input_schema": {
            "type": "object",
            "properties": {
                "entity":            {"type": "string", "description": "Starting entity name"},
                "relationship_type": {"type": "string", "description": "Optional: filter by relationship type"}
            },
            "required": ["entity"]
        }
    },
    {
        "name": "fetch_live_price",
        "description": "Fetch the current live price for a ticker from IBKR. Always use this for price-sensitive questions — never rely on indexed data for prices.",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Ticker symbol e.g. NVDA"}
            },
            "required": ["ticker"]
        }
    }
]


# Mock tool handlers
def handle_tool(name: str, inputs: dict) -> str:
    if name == "search_portfolio":
        return (
            "NVDA: 50 shares, avg cost $620.00, current (indexed) $875.40, P&L +41.19%. "
            "Technology sector. Position value at index time: $43,770."
        )
    elif name == "search_market_news":
        return (
            "Barclays reiterates NVDA Overweight, price target $1,050. Blackwell demand strong. "
            "Risk: export controls on H20 could cut FY2027 revenue by $4B."
        )
    elif name == "traverse_knowledge_graph":
        entity = inputs.get("entity", "")
        paths = graph_traverse(entity, G, max_hops=2)
        if not paths:
            return f"No graph paths found from '{entity}'."
        lines = [f"{p['from']} --{p['relationship']}--> {p['to']}" for p in paths[:5]]
        return "Graph paths: " + " | ".join(lines)
    elif name == "fetch_live_price":
        ticker = inputs.get("ticker", "")
        # Simulated live prices (fresh, not stale)
        prices = {"NVDA": 920.40, "AAPL": 214.30, "MSFT": 428.10,
                  "GOOGL": 180.50, "JPM": 223.80}
        price = prices.get(ticker.upper(), None)
        if price:
            return f"{ticker.upper()} live price: ${price:.2f} (fetched fresh from IBKR at query time)"
        return f"Ticker {ticker} not found in account."
    return "Unknown tool."


# Agentic RAG loop
def agentic_rag(user_question: str, max_turns: int = 5) -> str:
    messages = [{"role": "user", "content": user_question}]
    tool_trace = []

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=800,
            system=(
                "You are a financial assistant with access to tools. "
                "Use tools to gather information before answering. "
                "For price-sensitive questions always call fetch_live_price — never trust indexed prices. "
                "Think step by step about which tools are needed."
            ),
            tools=TOOLS,
            messages=messages
        )

        if response.stop_reason == "end_turn":
            # Extract text from the response
            final = next((b.text for b in response.content if hasattr(b, "text")), "")
            return final, tool_trace

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = handle_tool(block.name, block.input)
                    tool_trace.append({"tool": block.name, "inputs": block.input, "result": result})
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            # Append assistant message + tool results
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

    return "Max turns reached.", tool_trace


question = "How does current Fed policy affect my NVDA position and should I be concerned?"
print("=" * 70)
print("AGENTIC RAG — full reasoning trace")
print("=" * 70)
print(f"Question: {question}\n")

final_answer, trace = agentic_rag(question)

print("Tools called (in order):")
for i, step in enumerate(trace, 1):
    print(f"  {i}. {step['tool']}({json.dumps(step['inputs'])})")
    print(f"     → {step['result'][:100]}...")

print(f"\nFinal answer:\n{final_answer}")

In agentic RAG, retrieval is intentional: the agent called `fetch_live_price` because it knew the indexed price would be stale for a position-valuation question, and `traverse_knowledge_graph` because the Fed→Tech→NVDA chain is a relationship question, not a similarity question. Your FinMentor critic agent becomes a retrieval quality assessor that gates whether generation should happen at all.

---
## Section 8 — When to Use Each Architecture

GraphRAG is not always better. The choice depends on whether relationships exist and whether you can afford the extraction overhead.

In [ ]:
def choose_rag_architecture(
    corpus_size: int,
    has_relationships: bool,
    query_type: str,        # "exact_fact" | "multi_hop" | "similarity" | "real_time"
    update_frequency: str   # "static" | "daily" | "real_time"
) -> tuple[str, str]:
    """
    Returns (architecture, reasoning).
    Priority: real-time needs > relationship needs > scale needs > default.
    """
    if update_frequency == "real_time":
        return (
            "Agentic RAG with live tool calls",
            "Indexed data is always stale for real-time queries — agent fetches fresh at query time"
        )
    if not has_relationships:
        return (
            "Standard hybrid RAG",
            "No entity relationships in corpus — graph extraction overhead not justified"
        )
    if query_type == "multi_hop":
        return (
            "GraphRAG",
            "Multi-hop queries require traversing entity relationships — vector similarity cannot assemble chains"
        )
    if corpus_size > 100_000:
        return (
            "GraphRAG with community summarization",
            "Large corpus benefits from hierarchical graph communities — pre-summarized clusters reduce retrieval search space"
        )
    return (
        "Standard hybrid RAG with CRAG",
        "Relationships present but queries are direct — hybrid retrieval with corrective loop handles edge cases"
    )


scenarios = [
    {
        "name": "FinMentor portfolio queries",
        "corpus_size": 5_000,
        "has_relationships": True,
        "query_type": "multi_hop",
        "update_frequency": "daily"
    },
    {
        "name": "Customer support FAQ",
        "corpus_size": 2_000,
        "has_relationships": False,
        "query_type": "similarity",
        "update_frequency": "static"
    },
    {
        "name": "Medical drug interaction database",
        "corpus_size": 500_000,
        "has_relationships": True,
        "query_type": "multi_hop",
        "update_frequency": "daily"
    },
    {
        "name": "Real-time market news analysis",
        "corpus_size": 50_000,
        "has_relationships": True,
        "query_type": "multi_hop",
        "update_frequency": "real_time"
    }
]

print("=" * 80)
print("ARCHITECTURE DECISION FRAMEWORK")
print("=" * 80)
for s in scenarios:
    arch, reason = choose_rag_architecture(
        s["corpus_size"], s["has_relationships"],
        s["query_type"],  s["update_frequency"]
    )
    print(f"\nScenario  : {s['name']}")
    print(f"Architecture: {arch}")
    print(f"Reasoning : {reason}")

---
## Section 9 — Key Observations

1. **Vector search finds similar things. Graphs find connected things.** These are different questions — choose your tool accordingly. A corpus of independent documents with no meaningful entity relationships gains nothing from graph extraction overhead.

2. **Entity resolution is GraphRAG's clearest win over vector search.** One node, all references, resolved once at index time. At query time, every variant of a name finds the same node and surfaces all source documents. Vector search requires every variant to independently score well against the query.

3. **CRAG's self-assessment loop is what makes RAG safe for high-stakes domains.** Catching bad retrieval before bad generation is the architectural equivalent of the adversarial pass rate metric from the evaluation notebook — both are about knowing when you don't know.

4. **Agentic RAG shifts retrieval from automatic to intentional.** The agent decides when and what to retrieve based on the query. This is what makes `fetch_live_price` a first-class tool: the agent calls it explicitly for price-sensitive questions rather than relying on indexed data that may be minutes or hours stale.

5. **GraphRAG is not always better.** Independent document corpora with no meaningful entity relationships — customer support FAQs, product documentation, static knowledge bases — gain nothing from graph extraction overhead. The decision is driven by whether relationships exist and whether queries need to traverse them.

In [ ]:
print("Commit message:")
print("  04D: GraphRAG — knowledge graphs, entity resolution, CRAG, agentic RAG with tool use")
print()
print("Push command:")
print("  git add 04-rag-systems/")
print("  git commit -m '04D: GraphRAG — knowledge graphs, entity resolution, CRAG, agentic RAG with tool use'")
print("  git push origin main")